In [1]:
# ==========================================
# 1. INSTALL & IMPORT LIBRARY
# ==========================================
!pip -q install gradio transformers pandas

In [2]:
import os
import re
import itertools
import pandas as pd
import torch
import torch.nn.functional as F
import gradio as gr
from transformers import (
    pipeline,
    BertTokenizerFast,
    BertForTokenClassification,
    BertForSequenceClassification,
    BertForQuestionAnswering
)

In [3]:
# ============================================================
# 2. PERSIAPAN GIT LFS & UNDUH REPOSITORI (JIKA DI COLAB)
# ============================================================

# Mengecek apakah folder 'models' sudah ada. Jika belum, lakukan git clone LFS.
if not os.path.exists("./models"):
    print("Mengunduh model IndoRPL (NER, RE, QA) via Git LFS...")
    !apt-get install git-lfs -y -q
    !git lfs install
    !git clone https://github.com/nabilars31/indorpl-ner-re-qa-prototype.git temp_repo
    !cp -r temp_repo/models ./models
    !rm -rf temp_repo
    print("Model berhasil diunduh dan disiapkan!")
else:
    print("Folder models ditemukan, siap memuat model.")

Mengunduh model IndoRPL (NER, RE, QA) via Git LFS...
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 3,544 kB of archives.
After this operation, 10.5 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 git-lfs amd64 3.0.2-1ubuntu0.3 [3,544 kB]
Fetched 3,544 kB in 2s (1,630 kB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.0.2-1ubuntu0.3_amd64.deb ...
Unpacking git-lfs (3.0.2-1ubuntu0.3) ...
Setting up git-lfs (3.0.2-1ubuntu0.3) ...
Processing triggers for man-db (2.10.2-1) ...
Git LFS initialized.
Cloning into 'temp_repo'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (22/22), done.

In [4]:
# ==========================================
# 3. LOAD SEMUA MODEL
# ==========================================
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

# --- Load NER ---
print("Memuat Model NER...")
path_ner = "./models/indorpl-ner"
tokenizer_ner = BertTokenizerFast.from_pretrained(path_ner)
model_ner = BertForTokenClassification.from_pretrained(path_ner)
ner_pipeline = pipeline("ner", model=model_ner, tokenizer=tokenizer_ner, aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)

# --- Load RE ---
print("Memuat Model RE...")
path_re  = "./models/indorpl-re"
tokenizer_re = BertTokenizerFast.from_pretrained(path_re)
model_re = BertForSequenceClassification.from_pretrained(path_re)
re_pipeline = pipeline("text-classification", model=model_re, tokenizer=tokenizer_re, truncation=True, max_length=512, device=0 if torch.cuda.is_available() else -1)

# --- Load QA ---
print("Memuat Model QA...")
path_qa  = "./models/indorpl-qa"
tokenizer_qa = BertTokenizerFast.from_pretrained(path_qa)
model_qa = BertForQuestionAnswering.from_pretrained(path_qa)
model_qa.to(device)
model_qa.eval()

print("Semua model berhasil dimuat!")

Menggunakan device: cuda:0
Memuat Model NER...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Memuat Model RE...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Memuat Model QA...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Semua model berhasil dimuat!


In [5]:
# ==========================================
# 4. FUNGSI PENDUKUNG
# ==========================================
def build_pairs_and_mark_text(text, entities):
    pairs_data = []
    for e1, e2 in itertools.permutations(entities, 2):
        if e1['start'] < e2['end'] and e2['start'] < e1['end']:
            continue

        positions = [
            (e1['start'], e1['end'], '[E1]', '[/E1]'),
            (e2['start'], e2['end'], '[E2]', '[/E2]')
        ]
        positions.sort(key=lambda x: x[0], reverse=True)

        marked_text = text
        for start, end, mark_start, mark_end in positions:
            marked_text = marked_text[:end] + f" {mark_end} " + marked_text[end:]
            marked_text = marked_text[:start] + f" {mark_start} " + marked_text[start:]

        marked_text = re.sub(r'\s+', ' ', marked_text).strip()

        pairs_data.append({
            'e1_text': e1['word'], 'e1_label': e1.get('entity_group', e1.get('entity', '-')),
            'e2_text': e2['word'], 'e2_label': e2.get('entity_group', e2.get('entity', '-')),
            'marked_abstract': marked_text
        })
    return pd.DataFrame(pairs_data)

In [6]:
def qa_inference(question_text, context_text):
    inputs = tokenizer_qa(
        question_text, context_text, return_tensors="pt", max_length=384,
        truncation="only_second", return_offsets_mapping=True, padding="max_length"
    )
    offset_mapping = inputs.pop("offset_mapping")[0]
    sequence_ids = inputs.sequence_ids(0)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_qa(**inputs)

    start_probs = F.softmax(outputs.start_logits, dim=-1)[0]
    end_probs = F.softmax(outputs.end_logits, dim=-1)[0]

    start_conf, start_idx = torch.max(start_probs, dim=-1)
    end_conf, end_idx = torch.max(end_probs, dim=-1)

    start_idx, end_idx = start_idx.item(), end_idx.item()
    confidence_score = (start_conf.item() + end_conf.item()) / 2

    if start_idx == 0 and end_idx == 0: return "[TIDAK ADA JAWABAN / NEGATIVE SAMPLE]", confidence_score
    if start_idx > end_idx: return "[PREDIKSI TIDAK VALID (Awal > Akhir)]", confidence_score
    if sequence_ids[start_idx] != 1 or sequence_ids[end_idx] != 1: return "[PREDIKSI BERADA DI LUAR AREA KONTEKS]", confidence_score

    try:
        char_start = offset_mapping[start_idx][0].item()
        char_end = offset_mapping[end_idx][1].item()
        final_answer = context_text[char_start:char_end]
        if not final_answer.strip(): return "[JAWABAN KOSONG / FORMAT TIDAK COCOK]", confidence_score
        return final_answer.strip(), confidence_score
    except Exception as e:
        return f"[ERROR SAAT SLICING TEKS: {e}]", confidence_score

In [7]:
# ==========================================
# 5. INTEGRASI NLP PIPELINE (NER, RE, QA)
# ==========================================
def process_nlp_pipeline(teks_input, pertanyaan_qa_multiline, progress=gr.Progress()):
    teks_input = teks_input.lower()

    # Template output kosong jika tidak ada teks
    df_empty_ner = pd.DataFrame(columns=["No", "Entitas", "Tipe Entitas", "Confidence Score"])
    df_empty_re = pd.DataFrame(columns=["No", "Entitas 1", "Relasi", "Entitas 2", "Confidence Score"])

    if not teks_input.strip():
        return "**Jumlah Entitas:** 0", df_empty_ner, "**Jumlah Relasi:** 0", df_empty_re, "Masukkan teks terlebih dahulu."

    # --- LANGKAH A: NER ---
    progress(0.1, desc="Langkah 1/3: Mengekstraksi Entitas (NER)...")
    raw_ner = ner_pipeline(teks_input)

    data_ner = []
    for i, item in enumerate(raw_ner, 1):
        entitas = item.get('word', '-')
        tipe = item.get('entity_group', item.get('entity', '-'))
        score = item.get('score', 0.0)
        data_ner.append([i, entitas, tipe, f"{score:.4f}"])

    df_ner = pd.DataFrame(data_ner, columns=["No", "Entitas", "Tipe Entitas", "Confidence Score"])
    keterangan_ner = f"### **Jumlah Entitas Terdeteksi:** {len(data_ner)}"

    # --- LANGKAH B: RE ---
    progress(0.4, desc="Langkah 2/3: Menganalisis Relasi Antar Entitas (RE)...")
    data_re = []

    if len(raw_ner) >= 2:
        df_pairs = build_pairs_and_mark_text(teks_input, raw_ner)
        if not df_pairs.empty:
            raw_results = []
            total_pairs = len(df_pairs)

            for index, row in df_pairs.iterrows():
                progress(0.4 + (0.3 * (index / total_pairs)), desc=f"Memproses relasi ke-{index+1} dari {total_pairs}...")

                marked_text = row['marked_abstract']
                prediction = re_pipeline(marked_text)[0]

                if prediction['label'] != 'TIDAK_ADA_RELASI':
                    raw_results.append({
                        'e1': row['e1_text'].strip(), 'e1_label': row['e1_label'],
                        'relation': prediction['label'],
                        'e2': row['e2_text'].strip(), 'e2_label': row['e2_label'],
                        'score': prediction['score']
                    })

            if raw_results:
                df_results = pd.DataFrame(raw_results).sort_values(by='score', ascending=False)
                df_unique = df_results.drop_duplicates(subset=['e1', 'relation', 'e2'], keep='first')

                # Memasukkan ke dalam list data untuk Dataframe
                for i, (idx, row) in enumerate(df_unique.iterrows(), 1):
                    data_re.append([i, row['e1'], row['relation'], row['e2'], f"{row['score']:.4f}"])

    df_re = pd.DataFrame(data_re, columns=["No", "Entitas 1", "Relasi", "Entitas 2", "Confidence Score"])
    keterangan_re = f"### **Jumlah Relasi Terdeteksi:** {len(data_re)}"

    # --- LANGKAH C: QA (Multi-Pertanyaan) ---
    progress(0.8, desc="Langkah 3/3: Mencari Jawaban untuk Pertanyaan (QA)...")
    output_qa = ""

    # Memisahkan input QA berdasarkan baris baru (Enter)
    daftar_pertanyaan = [q.strip() for q in pertanyaan_qa_multiline.split('\n') if q.strip()]

    if not daftar_pertanyaan:
        output_qa = "Pertanyaan kosong. Masukkan minimal 1 pertanyaan untuk melihat hasil QA."
    else:
        for urutan, q in enumerate(daftar_pertanyaan, 1):
            jawaban, skor = qa_inference(q, teks_input)
            if skor < 0.01:
                jawaban = "[TIDAK ADA JAWABAN / CONFIDENCE TERLALU RENDAH]"

            output_qa += f"**[PERTANYAAN {urutan}]** {q}\n\n"
            output_qa += f"**Jawaban:** {jawaban}\n\n"
            output_qa += f"**Confidence Score:** {skor:.4f}\n\n"
            output_qa += "---\n\n"

    progress(1.0, desc="Selesai!")
    return keterangan_ner, df_ner, keterangan_re, df_re, output_qa

In [8]:
# ==========================================
# 6. RANCANGAN ANTARMUKA (GRADIO UI)
# ==========================================
with gr.Blocks(title="Prototipe NLP Terintegrasi") as demo:
    gr.Markdown("# Prototipe Sistem NLP Terintegrasi")
    gr.Markdown("Masukkan teks abstrak ilmiah untuk memproses NER, RE, dan QA secara otomatis.")

    with gr.Row():
        with gr.Column(scale=1):
            input_teks = gr.Textbox(
                label="Teks Input (Artikel/Abstrak)",
                lines=12,
                placeholder="Masukkan teks abstrak di sini..."
            )
            input_pertanyaan = gr.Textbox(
                label="Pertanyaan (Bisa lebih dari 1)",
                lines=5,
                placeholder="Apa solusi untuk...?\n(Tekan Enter untuk menambah pertanyaan kedua, dst.)"
            )
            btn_proses = gr.Button("Analisis Teks Abstrak", variant="primary")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("1. Hasil NER"):
                    info_ner = gr.Markdown("### **Jumlah Entitas Terdeteksi:** 0")
                    tabel_ner = gr.Dataframe(
                        headers=["No", "Entitas", "Tipe Entitas", "Confidence Score"],
                        datatype=["number", "str", "str", "number"],
                        interactive=False,
                        row_count=(10, "dynamic"),
                        wrap=True,
                        # Menentukan lebar presisi tiap kolom (Total 100%)
                        column_widths=["10%", "35%", "35%", "20%"]
                    )

                with gr.TabItem("2. Hasil RE"):
                    info_re = gr.Markdown("### **Jumlah Relasi Terdeteksi:** 0")
                    tabel_re = gr.Dataframe(
                        headers=["No", "Entitas 1", "Relasi", "Entitas 2", "Confidence Score"],
                        datatype=["number", "str", "str", "str", "number"],
                        interactive=False,
                        row_count=(10, "dynamic"),
                        wrap=True,
                        # Menentukan lebar presisi tiap kolom (Total 100%)
                        column_widths=["8%", "22%", "25%", "22%", "23%"]
                    )

                with gr.TabItem("3. Hasil QA"):
                    output_qa = gr.Markdown()

    btn_proses.click(
        fn=process_nlp_pipeline,
        inputs=[input_teks, input_pertanyaan],
        outputs=[info_ner, tabel_ner, info_re, tabel_re, output_qa]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b87c7a14be384de4ef.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
